In [0]:
# ======================================
# Dataset: olist_order_reviews
# Layer: Silver
# Source: Bronze (DELTA)
# Grain: 1 row per review_id
# ======================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format('delta').load(BRONZE_ORDER_REVIEWS_PATH)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("review_id"),
    F.col("order_id"),
    F.col("review_score").cast(IntegerType()).alias("review_score"),
    F.to_timestamp("review_creation_date").alias("review_creation_date"),
    F.to_timestamp("review_answer_timestamp").alias("review_answer_timestamp"),
    F.trim(F.col("review_comment_title")).alias("review_comment_title"),
    F.trim(F.col("review_comment_message")).alias("review_comment_message")
)

In [0]:
df = df.filter(
    F.col("review_id").isNotNull() &
    F.col("order_id").isNotNull()
)

In [0]:
df = df.filter(F.col("review_score").between(1, 5))

In [0]:
df = df.filter(
    (F.col("review_answer_timestamp").isNull()) |
    (F.col("review_creation_date") <= F.col("review_answer_timestamp"))
)


In [0]:
window = Window.partitionBy("review_id").orderBy(
    F.col("review_creation_date").desc()
)

df = (
    df
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)


In [0]:
df.printSchema()

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())

In [0]:
write_silver(df, SILVER_ORDER_REVIEWS_PATH)